## Marketing Mix Modeling with Google Meridian

As a complement to the OLS regression framework developed, we implement a
Bayesian Marketing Mix Model using **Google Meridian**.

While the OLS model gives us precise coefficient estimates with explicit control over
model structure (adstock transformations, AR lags, interaction terms), Meridian offers
a different set of advantages:

- **Automatic adstock and saturation estimation**: rather than fixing decay parameters
  via grid search, Meridian learns the shape of carryover and diminishing returns
  directly from the data
- **Full posterior uncertainty**: instead of point estimates, every parameter comes with
  a probability distribution, making uncertainty explicit
- **Built-in budget optimization**: once the model is fit, Meridian can directly solve
  for the spend allocation that maximizes trials under a given budget constraint



One important structural difference from the OLS model: Meridian does not include
autoregressive (AR) lags of the outcome variable.

This is standard practice in MMM:
adding lagged trials would confound the media attribution by absorbing variance that
belongs to the channels.

As a result, the $R^2$ will be lower than in OLS, which is
expected and not a concern for the purposes of media evaluation.

In [ ]:
# NOTE
# Designed for Google Colab with GPU (Runtime > Change runtime type > GPU)
# Required packages are installed below

!pip install google-meridian

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# os.chdir('/content/drive/My Drive/MKT')

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import meridian
from meridian.data import input_data as input_data_lib
from meridian.model import model as model_lib
from meridian.model import spec as model_spec_lib
from meridian.model import prior_distribution as prior_lib

from meridian.analysis import analyzer as analyzer_lib
from meridian.analysis import optimizer as optimizer_lib
#print("Meridian version:", meridian.__version__)

In [ ]:
# Inspect Meridian package structure
import pkgutil

for module in pkgutil.iter_modules(meridian.__path__):
    print(module.name)

# 1. Data Preparation

In [ ]:
# Upload the Excel file to Colab first (Files panel on the left)
df = pd.read_excel('IMA2026_AppTrials_DB.xlsx')

# Parse date and sort
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Clean column names (remove spaces)
df.rename(columns={
    'RADIO_Grp ': 'RADIO_Grp',
    'WEB MENTIONS': 'WEB_MENTIONS',
    'tot tv': 'tot_tv'
}, inplace=True)

print(df.shape)
print(df.dtypes)
df.head()

**Search split diagnostics**

In [ ]:
# 1. How many days was broad search actually active?
print("Broad search active days:", (df['search_broad_spend'] > 0).sum())
print("Always-on active days:", (df['search_alwayson_spend'] > 0).sum())

# 2. How much do they overlap / correlate?
print("\nCorrelation between broad and always-on clicks:")
print(df[['clic_search_alwayson', 'clic_search_broad']].corr())

# 3. Spend distribution
print("\nBroad spend stats:")
print(df['search_broad_spend'].describe())
print("\nAlways-on spend stats:")
print(df['search_alwayson_spend'].describe())

**InputData construction**

Meridian requires data to be structured as an `InputData` object built with **xarray**
arrays. Unlike a standard pandas DataFrame, xarray uses named dimensions, and Meridian
is strict about which dimension names it expects for each type of variable:

| Variable type | Expected dimension |
|---|---|
| KPI (trials) | `time` |
| Population | `time` |
| Control variables | `time` |
| Media exposure | `media_time` |
| Media spend | `time` |

This distinction between `time` and `media_time` exists because in some real-world
setups, media and KPI data may be observed at different frequencies. In our case they
are both daily, but the naming convention must still be respected.

We include **10 media channels**:
- **Online** (exposure + spend): Search (always-on + broad), CPC/CPM, CPD
- **Offline** (GRPs used as proxy for both exposure and spend): TV Paid, RAI TV
  flights 1–4, Radio

We include **8 control variables**: day-of-week dummies and other non-media factors
carried over from the OLS specification.

> ⚠️ **Limitation — offline spend inputs**: TV and Radio spend is expressed in GRPs
> (audience rating points), not euros. This is a data availability constraint. As a
> consequence, the ROI estimates and budget optimization outputs for TV and Radio
> channels should be interpreted with caution and are not directly comparable to the
> online channel figures.

In [ ]:
df['day_name'] = df['Date'].dt.day_name()
dow_dummies = pd.get_dummies(df['day_name'], prefix='dow', drop_first=False)
dow_dummies = dow_dummies.drop(columns=['dow_Monday'])  # Monday = reference
df = pd.concat([df, dow_dummies], axis=1)

control_cols = ['DUMMY_SEARCH', 'WEB_MENTIONS',
                'dow_Tuesday', 'dow_Wednesday', 'dow_Thursday',
                'dow_Friday', 'dow_Saturday', 'dow_Sunday']

In [ ]:
# For Meridian we need two arrays per channel:
# - media:       exposure metric (GRPs, clicks)
# - media_spend: cost metric (spend in currency)
# Where we only have one, we use the same column for both

# --- Online channels: we have both spend and clicks ---
media_exposure_cols = [
    'CPC/CPM_clic',               # exposure = clicks
    'clic_search_alwayson',        #
    'clic_search_broad',           #
    'CPD_CLIC',                    # exposure = clicks
    'TV_Grp_Mediaset_Premium_Sky', # exposure = GRPs
    'TV_Grp_Rai3_OnAir1',
    'TV_Grp_Rai3_OnAir2',
    'TV_Grp_Rai3_OnAir3',
    'TV_Grp_Rai3_OnAir4',
    'RADIO_Grp'
]

media_spend_cols = [
    'CPC/CPM_spend',
    'search_alwayson_spend',       #
    'search_broad_spend',          #
    'CPD_spend',
    'TV_Grp_Mediaset_Premium_Sky', # GRP proxy (no separate spend)
    'TV_Grp_Rai3_OnAir1',
    'TV_Grp_Rai3_OnAir2',
    'TV_Grp_Rai3_OnAir3',
    'TV_Grp_Rai3_OnAir4',
    'RADIO_Grp'
]

channel_names = [
    'CPC_CPM',
    'Search_AlwaysOn',
    'Search_Broad',
    'CPD',
    'TV_Paid', 'TV_RAI_1', 'TV_RAI_2', 'TV_RAI_3', 'TV_RAI_4',
    'Radio'
]

n_channels = len(channel_names)  # now 10 instead of 9
print(f"Channels ({n_channels}):", channel_names)

In [ ]:
from meridian.data.input_data import constants
print(constants.POSSIBLE_INPUT_DATA_ARRAY_NAMES)

In [ ]:
n_times = len(df)
geo    = ['Italy']
dates  = df['Date'].astype(str).values

kpi_da = xr.DataArray(
    df['Trial'].values.reshape(1, n_times).astype(float),
    dims=['geo', 'time'],
    coords={'geo': geo, 'time': dates},
    name='kpi'
)

population_da = xr.DataArray(
    np.array([1.0]),
    dims=['geo'],
    coords={'geo': geo},
    name='population'
)

controls_da = xr.DataArray(
    df[control_cols].values.reshape(1, n_times, len(control_cols)).astype(float),
    dims=['geo', 'time', 'control_variable'],
    coords={'geo': geo, 'time': dates, 'control_variable': control_cols},
    name='controls'
)

media_da = xr.DataArray(
    df[media_exposure_cols].values.reshape(1, n_times, n_channels).astype(float),
    dims=['geo', 'media_time', 'media_channel'],
    coords={'geo': geo, 'media_time': dates, 'media_channel': channel_names},
    name='media'
)

spend_da = xr.DataArray(
    df[media_spend_cols].values.reshape(1, n_times, n_channels).astype(float),
    dims=['geo', 'time', 'media_channel'],
    coords={'geo': geo, 'time': dates, 'media_channel': channel_names},
    name='media_spend'
)

for da in [kpi_da, population_da, controls_da, media_da, spend_da]:
    print(f"name={da.name}, shape={da.shape}")

input_data_obj = input_data_lib.InputData(
    kpi=kpi_da,
    kpi_type='non_revenue',
    media=media_da,
    media_spend=spend_da,
    controls=controls_da,
    population=population_da,
)
print("\nInputData created successfully")

# 2. Run the model

## 2.1. Model Specification and Sampling

We instantiate the Meridian model using the `InputData` object constructed above,
with no additional manual transformations — adstock and saturation curves are estimated
internally by the model.

Model estimation is performed via **MCMC sampling**. Internally, Meridian uses the
**No-U-Turn Sampler (NUTS)** by default — an efficient gradient-based algorithm that
handles correlated parameters well, which is typical in MMM settings. This happens
under the hood and is not something we configure explicitly.

What we do specify is the sampling configuration:

| Parameter | Value |
|---|---|
| Chains | 2 |
| Adaptation steps | 500 |
| Burn-in steps | 500 |
| Retained samples | 1,000 |
| Random seed | 42 |



The two chains run independently and are used to diagnose convergence: if they explore
the posterior distribution consistently, we can trust the results.

Convergence is
assessed via the **R-hat statistic**: values close to 1.0 indicate the chains have
mixed well.

In [ ]:
model_spec = model_spec_lib.ModelSpec(
    # Adstock: Meridian estimates decay per channel automatically
    # Saturation: Hill function estimated per channel
    # All priors are Meridian defaults — no manual tuning needed
)

mmm = model_lib.Meridian(
    input_data=input_data_obj,
    model_spec=model_spec
)

print("Model ready")

In [ ]:
mmm.sample_posterior(
    n_chains=2,
    n_adapt=500,
    n_burnin=500,
    n_keep=1000,
    seed=42
)

print("Sampling complete")

## 2.2. Post-Sampling Analysis: The Analyzer Object

Once sampling is complete, results are accessed through Meridian's `Analyzer` class,
which wraps the fitted model and exposes methods for extracting key metrics.

In [ ]:
analyzer = analyzer_lib.Analyzer(mmm)

# Inspect available methods on the analyzer object
print(dir(analyzer))

In [ ]:
print([m for m in dir(analyzer) if not m.startswith('_')])

In [ ]:
analyzer.rhat_summary()

All parameters show R-hat values below 1.02, and no parameter has a flagged sample (`percent_bad_rhat` = 0). This confirms that both chains converged to the same posterior distribution.

# 3. Results and Diagnostics

## 3.1. Model fit



**Predictive accuracy**: standard fit statistics including
- $R^2$,
- MAPE (Mean Absolute Percentage Erro), and
- wMAPE (weighted MAPE),

computed against the observed trial data.



In [ ]:
# 1. Model fit summary
print("Predictive Accuracy")
print(analyzer.predictive_accuracy())

In [ ]:
# prettier print
analyzer.predictive_accuracy().to_dataframe()

As discussed, the $R^2$ is lower than the OLS model due
to the absence of autoregressive lags, which is expected in the MMM framework.

## 3.2. Channel ROI

**ROI per channel**:
- Because Meridian is Bayesian, ROI is not a single number but
a distribution: one value per posterior sample (1,000 in our case).
- The raw output
is therefore a tensor of shape (chains × samples × channels).

- Summary statistics
(mean, credible intervals) are extracted in the next step.

In [ ]:
# 2. ROI per channel
print("ROI per Channel")
print(analyzer.roi())

**Marginal ROI**:
- The return on the last euro spent in each channel, as opposed to
the average return over total spend.
- Due to diminishing returns, marginal ROI is
always lower than average ROI.

  This is the more actionable metric for budget reallocation decisions: it tells you where an additional euro would be most efficiently deployed.

In [ ]:
# 3. Marginal ROI
print("Marginal ROI")
print(analyzer.marginal_roi())

We first sample the prior distribution, which Meridian requires to populate the prior columns in `summary_metrics()`.

The raw output is an xarray Dataset. The structure shows available dimensions and variables before extraction

In [ ]:
mmm.sample_prior(n_draws=500, seed=42)
print(analyzer.summary_metrics())

The full summary is first rendered as a DataFrame for inspection.

In [ ]:
analyzer.summary_metrics().to_dataframe()

We retain only the posterior distribution, discarding the prior, and extract the four metrics of interest.

The filtered table is pivoted so that channels become rows and metrics become columns, giving a compact view of each channel's performance.

In [ ]:
summary = analyzer.summary_metrics().to_dataframe().reset_index()

# Keep only posterior distribution
summary_post = summary[summary['distribution'] == 'posterior']

# Keep only the metrics we care about
key_metrics = ['incremental_outcome', 'pct_of_contribution', 'roi', 'mroi']

summary_clean = summary_post[['channel', 'metric'] + key_metrics]

# Pivot so metrics become columns
summary_pivot = summary_clean.pivot(index='channel', columns='metric', values=key_metrics)

# Flatten multi-level columns
summary_pivot.columns = ['_'.join(col) for col in summary_pivot.columns]

# Keep mean and credible interval only
cols_to_keep = [c for c in summary_pivot.columns if any(x in c for x in ['mean', 'ci_lo', 'ci_hi'])]
summary_pivot = summary_pivot[cols_to_keep]

# Now rename for display
summary_pivot.columns = [col.replace('_', ' ') for col in summary_pivot.columns]
summary_pivot

In [ ]:
# Total observed trials over the sample period
total_observed = df['Trial'].sum()
print(f"Total observed trials: {total_observed:,.0f}")
print(f"Total incremental outcome (all channels): 26,412")
print(f"Implied baseline: {total_observed - 26412:,.0f}")

> `pct of contribution`: Values above 100% arise when the estimated baseline is negative; media contributions are not capped at total KPI.

Channels are sorted by posterior mean ROI to facilitate comparison.

In [ ]:
roi_summary = summary_pivot[['roi mean', 'roi ci lo', 'roi ci hi']].copy()
roi_summary = roi_summary.sort_values('roi mean', ascending=False)
roi_summary.round(4)

Channel-level ROI estimates (posterior mean with 95% credible interval) are extracted
from the posterior summary and sorted by mean ROI.

**Search Always-On** dominates by a wide margin.
  - It generates roughly one additional trial for every four euros spent. This is the highest-ROI channel by far, and the credible interval is tight relative to its magnitude, reflecting strong posterior identification.
  - This is consistent with the OLS findings, where search was the largest and most significant contributor to trials.


**Search Broad** is markedly weaker.
- This is notable: despite being activated during the high-demand November period, broad search does not appear to generate incremental trials at a rate comparable to always-on.
- One interpretation is that the burst activation of broad search captures demand already created by TV and always-on. In other words, it may be picking up existing intent rather than creating new demand. This is consistent with the second-screen mechanism documented in the OLS model.

**CPC/CPM** ranks third, with the tightest credible interval of any channel, reflecting the large and consistent spend history that gives the model a lot of data to work with.

**TV and Radio** ROI figures should not be interpreted at face value.
- Their spend inputs are expressed in GRPs rather than euros, making the absolute ROI values non-comparable to online channels.
- The near-zero contribution shares for TV in the contribution chart reflect this identification challenge in Meridian, not necessarily an absence of effect, which the OLS model documents clearly.

In [ ]:
summary = analyzer.summary_metrics()

channels = summary.coords['channel'].values[:-1]  # exclude 'All Channels'

# Extract posterior contributions and ROI
pct_contrib = summary['pct_of_contribution'].sel(
    distribution='posterior', metric='median').values[:-1]
roi_median = summary['roi'].sel(
    distribution='posterior', metric='median').values[:-1]
roi_lo = summary['roi'].sel(
    distribution='posterior', metric='ci_lo').values[:-1]
roi_hi = summary['roi'].sel(
    distribution='posterior', metric='ci_hi').values[:-1]
spend_pct = summary['pct_of_spend'].values[:-1]

# --- Plot 1: Channel Contribution ---
fig, axs = plt.subplots(1,3, figsize=(18, 5))
ax = axs[0]
bars = ax.barh(channels, pct_contrib, color='steelblue')
ax.set_xlabel('% Contribution to App Trials (posterior median)')
ax.set_title('Channel Contribution — Meridian Bayesian MMM')
for bar, val in zip(bars, pct_contrib):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

# --- Plot 2: ROI with uncertainty ---
ax = axs[1]
y = range(len(channels))
ax.barh(y, roi_median, xerr=[roi_median - roi_lo, roi_hi - roi_median],
        color='steelblue', alpha=0.7, capsize=4)
ax.set_yticks([])
#ax.set_yticklabels(channels)
ax.set_xlabel('ROI (trials per unit spend) — posterior median + 95% CI')
ax.set_title('ROI by Channel — Meridian Bayesian MMM')
ax.axvline(0, color='black', linewidth=0.8)


# --- Plot 3: Spend vs Contribution comparison ---
ax = axs[2]
x = np.arange(len(channels))
w = 0.35
ax.bar(x - w/2, spend_pct, w, label='% of Spend', color='coral', alpha=0.8)
ax.bar(x + w/2, pct_contrib, w, label='% of Contribution', color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(channels, rotation=30, ha='right')
ax.set_ylabel('%')
ax.set_title('Spend vs Contribution by Channel')
ax.legend()

plt.tight_layout()
plt.show()

- The contribution chart (left) shows Search Always-On driving the vast majority of media-attributed trials, with CPC/CPM a distant second and all other channels negligible.

- The ROI chart (centre) confirms Search Always-On as the highest-return channel by a wide margin. The scale compression makes other channels' uncertainty bands appear flat, but CPC/CPM stands out for its unusually tight credible interval, a consequence of its large and stable spend history.

- The spend vs contribution chart (right) highlights the most actionable imbalance: CPC/CPM absorbs over 70% of the budget while generating roughly 20% of contribution, a clear sign of over-investment relative to its productivity.

## 3.3. Response Curves

In [ ]:
response = analyzer.response_curves()
print(response)

In [ ]:
response.to_dataframe()

In [ ]:
# Check what's available first
from meridian.analysis import optimizer as optimizer_lib
print([m for m in dir(optimizer_lib) if not m.startswith('_')])

In [ ]:
# Convert to DataFrame first
response = analyzer.response_curves()
response_df = response.to_dataframe().reset_index()
print(response_df.columns.tolist())
print(response_df.head())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(25, 8))
axes = axes.flatten()

for i, channel in enumerate(channel_names):
    ch = response_df[response_df['channel'] == channel]

    mean = ch[ch['metric'] == 'mean']
    ci_lo = ch[ch['metric'] == 'ci_lo']
    ci_hi = ch[ch['metric'] == 'ci_hi']

    axes[i].plot(mean['spend_multiplier'], mean['incremental_outcome'],
                 color='steelblue', label='mean')
    axes[i].fill_between(mean['spend_multiplier'],
                         ci_lo['incremental_outcome'].values,
                         ci_hi['incremental_outcome'].values,
                         alpha=0.3, color='steelblue', label='90% CI')
    axes[i].set_title(channel)
    axes[i].set_xlabel('Spend multiplier')
    axes[i].set_ylabel('Incremental trials')
    axes

plt.tight_layout()
plt.show()

Response curves show how incremental trials would change if each channel's spend were scaled between 0× and 2× its historical level, holding all else constant. The shape of each curve reflects Meridian's estimated saturation function for that channel: the flattening at higher spend multipliers indicates diminishing returns.

- Search Always-On and CPC/CPM show the steepest curves, confirming they are the most productive channels and that both still have room to scale before hitting saturation.

- Search Broad and CPD curves are considerably flatter, consistent with their low ROI estimates. TV and Radio curves are near-zero, which reflects the GRP spend limitation rather than a structural absence of effect.

The shaded band represents the 90% credible interval. Wider bands, visible particularly for TV and Radio, indicate greater posterior uncertainty about the true shape of the response, driven by sparse activation and limited spend variation in the data.

## 3.4 Adstock Decay

In [ ]:
decay = analyzer.adstock_decay()
decay

The decay table reports, for each channel, how much of the original effect remains after each time unit.
- A value of 0.72 at 0.2 days for CPC/CPM, for example, means roughly 72% of the impact persists into the following period.
- Channels with slower decay retain their effect longer, meaning past spend continues to contribute to current trials.

In [ ]:
# --- Adstock Decay Curves ---
decay = analyzer.adstock_decay()
decay_post = decay[decay['distribution'] == 'posterior']

channels_decay = decay_post['channel'].unique()
fig, axes = plt.subplots(2, 5, figsize=(25, 8))
axes = axes.flatten()

for i, channel in enumerate(channels_decay):
    ch = decay_post[decay_post['channel'] == channel]
    axes[i].plot(ch['time_units'], ch['mean'], color='steelblue', label='mean')
    axes[i].fill_between(ch['time_units'], ch['ci_lo'], ch['ci_hi'],
                         alpha=0.3, color='steelblue', label='90% CI')
    axes[i].set_title(channel)
    axes[i].set_xlabel('Days')
    axes[i].set_ylabel('Remaining effect')
    axes[i].set_ylim(0, 1)
    axes[i].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)

plt.suptitle('Adstock Decay by Channel — Posterior Estimates', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Adstock decay curves show how much of a channel's effect persists over time after an exposure, estimated directly from the data rather than fixed manually as in the OLS model.

- Online channels, CPC/CPM and Search, decay rapidly, with most of the effect dissipating within 1-2 days, consistent with the immediate nature of performance digital advertising.
- All other channels, CPD, Radio, and TV, show slower decay but with substantially wider credible intervals, reflecting limited activation variation in the data.

  TV and Radio uncertainty is the most pronounced, active only during the November–December burst.

# 4. Budget Optimization

Meridian's built-in optimizer takes the estimated response curves and finds the spend allocation that maximises total incremental trials under a fixed budget constraint.

We set the budget equal to the total historical spend, so the exercise is a reallocation problem rather than a budget expansion.

> The question is whether the same money, deployed differently, would generate more trials.


Spend constraints are set at ±30% of each channel's historical allocation, preventing the optimizer from recommending implausibly large shifts that the response curves may not reliably support at the extremes.

In [ ]:
# Total historical spend
total_budget = float(df[media_spend_cols].values.sum())
print(f"Total historical budget: {total_budget:,.0f}")

optimizer = optimizer_lib.BudgetOptimizer(analyzer)


In [ ]:
import inspect
print(inspect.signature(optimizer_lib.FixedBudgetScenario.__init__))
print(inspect.signature(optimizer_lib.BudgetOptimizer.__init__))

In [ ]:
print([m for m in dir(optimizer) if not m.startswith('_')])
print(inspect.signature(optimizer_lib.BudgetOptimizer.optimize))

In [ ]:
optimizer = optimizer_lib.BudgetOptimizer(meridian=mmm)

results = optimizer.optimize(
    fixed_budget=True,
    budget=total_budget,
    use_kpi=True
)

print(results)

In [ ]:
# Extract non-optimized vs optimized data
non_opt = results._nonoptimized_data
opt = results._optimized_data

channels_opt = non_opt.coords['channel'].values

comparison_df = pd.DataFrame({
    'channel': channels_opt,
    'current_spend': non_opt['spend'].values,
    'optimized_spend': opt['spend'].values,
    'current_contribution': non_opt['incremental_outcome'].sel(metric='mean').values,
    'optimized_contribution': opt['incremental_outcome'].sel(metric='mean').values,
    'current_pct_spend': non_opt['pct_of_spend'].values,
    'optimized_pct_spend': opt['pct_of_spend'].values,
})

comparison_df['spend_change'] = comparison_df['optimized_spend'] - comparison_df['current_spend']
comparison_df['spend_change_pct'] = (comparison_df['spend_change'] / comparison_df['current_spend'].replace(0, np.nan)) * 100

print(comparison_df[['channel','current_spend','optimized_spend','spend_change_pct']].round(1))
print(f"\nTotal incremental outcome — Current:   {non_opt.attrs['total_incremental_outcome']:,.0f} trials")
print(f"Total incremental outcome — Optimized: {opt.attrs['total_incremental_outcome']:,.0f} trials")
print(f"Gain from reallocation: {opt.attrs['total_incremental_outcome'] - non_opt.attrs['total_incremental_outcome']:,.0f} trials (+{((opt.attrs['total_incremental_outcome'] / non_opt.attrs['total_incremental_outcome']) - 1) * 100:.1f}%)")

In [ ]:
# --- Budget Allocation Chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x = np.arange(len(channels_opt))
w = 0.35

# Spend comparison
axes[0].bar(x - w/2, comparison_df['current_pct_spend'],
            w, label='Current', color='coral', alpha=0.8)
axes[0].bar(x + w/2, comparison_df['optimized_pct_spend'],
            w, label='Optimized', color='steelblue', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(channels_opt, rotation=30, ha='right')
axes[0].set_ylabel('% of Total Budget')
axes[0].set_title('Budget Allocation: Current vs Optimized')
axes[0].legend()

# Spend change
colors = ['green' if v >= 0 else 'red' for v in comparison_df['spend_change']]
axes[1].barh(channels_opt, comparison_df['spend_change'] / 1000,
             color=colors, alpha=0.7)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Spend Change (€ thousands)')
axes[1].set_title('Recommended Spend Change by Channel')

plt.suptitle('Meridian Budget Optimization Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

- Search Always-On and CPC/CPM receive increased budget, while CPD is cut substantially since its spend share was large relative to its contribution.
- Search Broad receives a modest increase, reflecting some residual productivity despite its lower ROI.

- - TV and Radio are unaffected, both because their historical input is expressed in GRPs rather than euros, making their spend non-comparable to the budget constraint. The NaN values in the table reflect this - they should be read as "unchanged," not "zero investment recommended.".

The net effect is a meaningful gain in projected trials from reallocation alone, without any increase in total budget.

# 5. Conclusion

The Meridian model corroborates the core findings of the OLS analysis while adding a layer of granularity and uncertainty quantification that point estimates cannot provide.

- Search Always-On is the dominant driver of app trials across both models, with a posterior ROI well above all other channels and a tight credible interval reflecting strong identification.
- CPC/CPM is a consistent second contributor but absorbs a disproportionate share of the budget relative to its productivity.
- CPD shows weak performance in both frameworks: the OLS model excluded it due to coefficient instability across specifications, and Meridian independently estimates it as the lowest-ROI online channel. This convergence across methods strengthens the recommendation to reallocate CPD budget toward higher-performing channels.
- TV and Radio, while significant in OLS, where their effect is captured through adstock transformations and an explicit interaction term, are not reliably identified in Meridian due to the GRP spend limitation, and their ROI estimates should not be compared directly to online channels.

The budget optimization exercise suggests meaningful efficiency gains are achievable through reallocation alone, primarily by shifting spend from CPD toward Search and CPC/CPM.

The two models are complementary by design.
- OLS provides a precise causal account of how TV drives search demand and ultimately trials, with explicit control over model structure.
- Meridian provides automatic adstock and saturation estimation, full posterior uncertainty, and actionable budget guidance.